In [24]:
import pandas as pd
from typing import Optional, List
from pydantic import BaseModel, Field
import re

In [25]:
# Load the uploaded Excel file to inspect its content
file_path = '../GEARBOx Ontology for LLM_In use.xlsx'
excel_data = pd.ExcelFile(file_path)

# Display the sheet names to understand the structure of the file
excel_data.sheet_names

['GEARBOx v4', 'xx-GEARBOx', 'xx-Example']

In [26]:
# Load the sheet to inspect its content
gearbox_v4_df = excel_data.parse('GEARBOx v4')
excel_data.close
# Display the first few rows of the dataframe to understand its structure
gearbox_v4_df.head()

,Parameter Name,Type,Description,Examples,MaxLength,Enum,ItemType,ItemDescription,ItemExamples,ItemEnum
0,age_min,integer,Does the eligibility criteria mention the mini...,NaN,10.0,NaN,NaN,NaN,NaN,NaN
1,age_max,integer,Does the eligibility criteria mention the maxi...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,weight_min,number,Does the eligibility criteria mention the mini...,NaN,10.0,NaN,NaN,NaN,NaN,NaN
3,weight_max,number,Does the eligibility criteria mention the maxi...,NaN,10.0,NaN,NaN,NaN,NaN,NaN
4,sex,string,Does the eligibility criteria mention sexes of...,NaN,15.0,"""Male"", ""Female""",NaN,NaN,NaN,NaN


In [27]:
# Function to generate class fields and enums
def generate_class_fields_and_enums(df, verb):
    fields = {}
    enum_classes = {}  # Dictionary to store generated enum classes as strings

    for _, row in df.iterrows():
        param_name = row['Parameter Name']
        param_name_exist = f"{param_name}_exist"

        # Determine the field type based on 'Type' and 'Enum' columns
        if row['Type'] in ['integer', 'number']:
            field_type = 'Optional[int]'
        elif pd.notna(row['Enum']) and 'Yes' in row['Enum']:
            field_type = 'Optional[Yes_no]'  # Assuming YesOrNo is a defined type elsewhere
        elif pd.notna(row['Enum']):
            field_type = f"Optional[List[{param_name.capitalize()}]]"
            
            # Create an enum class for this parameter
            enum_items = [item.strip().strip('"') for item in row['Enum'].split(',')]
            enum_class_str = f"class {param_name.capitalize()}(Enum):\n"
            for item in enum_items:
                if "(" in item and ")" in item:
                    enum_name = re.search(r"\((.*?)\)", item).group(1)
                    enum_name = enum_name.strip().replace(" ", "")
                    enum_name = enum_name.strip().replace("-", "").upper()
                else:
                    enum_name = item.strip().replace("-", "").replace(" ", "").upper()
                enum_class_str += f"    {enum_name} = \"{item}\"\n"
            
            enum_classes[param_name.capitalize()] = enum_class_str
            
        else:
            field_type = 'Optional[str]'
            

        # Combine description and examples if available
        description = row['Description']
        example = row['Examples']
        if pd.notna(example):
            description += f" Example: {example}"
        parts = description.split('?')
        first_yesno = parts[0] + '?'
        second_criteria = parts[1].strip().replace("{verb}", verb)
        
        fields[param_name_exist] = f"Yes_no = Field(..., description=\"{first_yesno}\")"
        fields[param_name] = f"{field_type} = Field(\n        None, \n        description=\"{second_criteria}\")"
    
    enum_items = ["Yes","No"]
    enum_class_str = f"class Yes_no(Enum):\n"
    for item in enum_items:
        enum_name = item.upper()
        enum_class_str += f"    {enum_name} = \"{item}\"\n"
    enum_classes["Yes_no"] = enum_class_str

    return fields, enum_classes


# Generate class fields from the DataFrame
class_fields_ic, enum_classes = generate_class_fields_and_enums(gearbox_v4_df, "included")
class_fields_ec, _ = generate_class_fields_and_enums(gearbox_v4_df, "excluded")

class_definition = "from enum import Enum\nfrom pydantic import BaseModel, Field\nfrom typing import Optional, List\n\n"
for enum_name, enum_definition in enum_classes.items():
    class_definition += f"{enum_definition}\n"

# Generating the ic class as a string
class_definition_ic = class_definition + "class Entity(BaseModel):\n"
for field_name, field_definition in class_fields_ic.items():
    class_definition_ic += f"    {field_name}: {field_definition}\n"

# Generating the ec class as a string
class_definition_ec = class_definition + "class Entity(BaseModel):\n"
for field_name, field_definition in class_fields_ec.items():
    class_definition_ec += f"    {field_name}: {field_definition}\n"

In [28]:
# Define the file path to save the class definition
output_IC_file_path = "Entity_IC.py"
output_EC_file_path = "Entity_EC.py"

# Save the class definition to a file
with open(output_IC_file_path, 'w') as file:
    file.write(class_definition_ic)

# Save the class definition to a file
with open(output_EC_file_path, 'w') as file:
    file.write(class_definition_ec)